# SwarmSort Gen 2, notebook 2: PSO search (Arm C)

Arm C tunes the six training hyperparameters (lr0, lrf, momentum, weight_decay, mosaic, scale) with the hand-written particle swarm optimizer in `src/pso.py`: **8 particles x 4 rounds = 32 proxy trainings** (seed 42), exactly as in Gen 1.

Every proxy training is 12 epochs on a stratified 40% subset of the Gen 2 train split at **imgsz 640**, and its val
mAP@50 is the fitness. The proxy-validity check is not repeated for Gen 2: Gen 1 showed that this proxy ranks
configurations like longer training (Spearman rho = 0.976, n = 8), and the recipe is unchanged.
The other search, `g2_3_search_random.ipynb`, has the same budget; **the two notebooks can run at the same time** in separate
Kaggle sessions (if Kaggle queues one, run them one after the other).

**Expected time: about 3.9 hours** on GPU T4 x2. Gen 1's PSO proxies took 214 s each on average; 214 s x 2.033 = 436 s (7.3 min) per proxy, x 32 = 13,940 s = 3.9 h.

Time estimates scale the measured Gen 1 times (GPU T4, imgsz 416, 7324 train images) by the pixel ratio (640/416)² = 2.367 and the train-image ratio 6290/7324 = 0.859, a factor of **2.033**. The expected 6290 Gen 2 train images come from the source research: Gen 1 subsample 3000 + 70% of TACO's 1500 (1050) + 70% of HITL's 3200 (2240). The real count is in `build_report.json`. The images are stored with a long side of up to 1280 px, so if JPEG decoding on Kaggle's 4 CPU cores limits the speed, allow up to about 30% more.

**Freeze protection.** The attached dataset is first copied to local disk (a few minutes; the time is printed), because two Gen 2 runs froze about 55 minutes in while reading it from the `/kaggle/input` mount. Every script runs under a watchdog (`src/watchdog_run.py`): after 20 minutes without output or file changes it is killed with its dataloader workers and restarted, at most 3 times, and the restart continues the work. **If a version stopped or froze, attach its output and run again: it continues.**

**Settings:** Accelerator = **GPU T4 x2** (one GPU is used, as in Gen 1), Internet = on (for the clone, pip and the
`yolov8n.pt` download). **Input:** Add Input -> Notebook Output -> **`g2_0_build_dataset`** (its finished version).
No dataset needs to be attached: the built images are inside that output. Run it with
**Save Version -> Save & Run All (Commit)** so it keeps running with the browser closed (up to 12 hours).

Outputs: `/kaggle/working/SwarmSort/results/gen2/search_pso.json` (every evaluation) and
`results/gen2/best_pso.json` (the best configuration, which the full training of Arm C
reads); proxy run folders go to `results/gen2/runs/`.

### Resuming after a dead or frozen session

**If a version stopped or froze, attach its output and run again: it continues.** `search_pso.json` is rewritten after every evaluation. On a restart `search.py` replays the search from its seed,
reuses every logged result instead of training again and continues at the first missing evaluation, so a dead session
loses at most one proxy training. To continue: open the notebook, **Add Input -> Notebook Output**, pick the previous
(failed or timed-out) version of this notebook (keep `g2_0_build_dataset` attached too), then run it again. The resume
cell copies that version's `results/gen2/search_pso.json` into place before the search starts. Do not change the
search command: a log made with other settings is refused. Within one session the watchdog does the same by itself:
after a 20-minute stall it restarts `search.py`, which replays the log.

In [ ]:
import os, subprocess, sys
REPO = "/kaggle/working/SwarmSort"
if not os.path.isdir(REPO):
    !git clone --depth 1 https://github.com/Shreyansh303/SwarmSort.git {REPO}
assert os.path.isdir(f"{REPO}/src"), "git clone failed: turn on Settings -> Internet and run again"
%cd {REPO}
# Kaggle's own CUDA build of torch is kept: only Ultralytics is installed, at the version pinned in requirements.txt
!pip install -q ultralytics==8.4.169
import torch, ultralytics
assert torch.cuda.is_available(), "No GPU: set Settings -> Accelerator -> GPU T4 x2 and run again"
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "|", torch.cuda.get_device_name(0))
if not os.path.isfile("src/watchdog_run.py"):
    raise RuntimeError("src/watchdog_run.py is not on GitHub yet: push the latest commits")
sys.path.insert(0, f"{REPO}/src")
from watchdog_run import run_watched
# About 10 s: check on this machine that a frozen script and its worker process are killed and retried
subprocess.run([sys.executable, "-m", "unittest", "-q",
                "tests.test_watchdog.TestWatchdog.test_b_stalled_script_is_killed_and_retried",
                "tests.test_watchdog.TestWatchdog.test_e_child_processes_are_killed_too"], check=True)


def run(*args, watch=(), stall_minutes=20, max_retries=3):
    """Run a repository script under the stall watchdog (src/watchdog_run.py). Its output is streamed here. If it
    prints nothing and nothing changes under `watch` for stall_minutes, it is killed together with its dataloader
    workers and started again with the same arguments, at most max_retries times (search.py then continues from its
    log, train_final.py --resume from last.pt). If the script stops with an error, this cell fails and the version
    is marked failed (as with check=True)."""
    args = [str(a) for a in args]
    print("$ python", " ".join(args), flush=True)
    run_watched([sys.executable, *args], watch=watch, stall_minutes=stall_minutes, max_retries=max_retries,
                env={**os.environ, "PYTHONUNBUFFERED": "1"})


In [ ]:
# Find the dataset built by g2_0_build_dataset (attached as input), copy it to this machine's local disk, and write
# configs that point at the local copy: /kaggle/working/gen2_cfg/data.yaml, data_test_<domain>.yaml, split.csv, lists/
# Why the copy: two Gen 2 runs froze about 55 minutes into training and sat idle until Kaggle's 12-hour limit,
# most likely because the attached notebook-output mount under /kaggle/input stopped answering. Training now reads
# only the local copy. It goes outside /kaggle/working, so it does not become part of this version's output.
# The copy reads that mount too, so it fails loudly (TimeoutError) if no file arrives for 2 minutes.
import glob, json, shutil
from watchdog_run import copy_files, list_tree
INPUT, CFG = "/kaggle/input", "/kaggle/working/gen2_cfg"
builds = sorted({os.path.dirname(p) for p in glob.glob(f"{INPUT}/**/build_report.json", recursive=True)
                 if os.path.isfile(os.path.join(os.path.dirname(p), "data.yaml"))
                 and os.path.isdir(os.path.join(os.path.dirname(p), "images"))})
if not builds:
    raise FileNotFoundError("The Generation 2 dataset is not attached. Add Input -> Notebook Output -> "
                            "g2_0_build_dataset (its finished version), then run again.")
if len(builds) > 1:
    raise RuntimeError("Several built datasets are attached; remove all but one version of g2_0_build_dataset:\n  "
                       + "\n  ".join(builds))
BUILD_SRC = builds[0]
print("Built dataset (attached):", BUILD_SRC)
files = list_tree(BUILD_SRC, exclude=["*.cache"])
size = sum(n for _, n in files)
for base in ("/kaggle/tmp", "/kaggle/temp", "/tmp"):  # local disk outside /kaggle/working
    if os.path.isdir(base) and os.access(base, os.W_OK) and shutil.disk_usage(base).free > 1.1 * size + 2e9:
        BUILD = f"{base}/gen2_data"
        break
else:
    raise OSError(f"No local folder has {size / 1e9:.1f} GB free for the dataset copy")
stamp = {"source": BUILD_SRC, "files": len(files), "bytes": size}
if os.path.isfile(f"{BUILD}/.copied.json") and json.load(open(f"{BUILD}/.copied.json")) == stamp:
    print("Local copy already made in this session:", BUILD)
else:
    shutil.rmtree(BUILD, ignore_errors=True)
    print(f"Copying {len(files)} files ({size / 1e9:.2f} GB) to {BUILD}", flush=True)
    info = copy_files(BUILD_SRC, BUILD, files, workers=16, stall_seconds=120)
    if list_tree(BUILD) != files:
        raise RuntimeError(f"The local copy {BUILD} differs from {BUILD_SRC} (file names or sizes)")
    json.dump(stamp, open(f"{BUILD}/.copied.json", "w"))
    print(f"Copied {info['files']} files, {info['bytes'] / 1e9:.2f} GB in {info['seconds']:.0f} s "
          f"({info['bytes'] / 1e6 / max(info['seconds'], 1):.0f} MB/s); names and sizes match the attached build")
for sub in ("images", "labels"):
    print(f"  {sub}/: {len(os.listdir(f'{BUILD}/{sub}'))} files")
if "--relocate-from" not in open("src/build_dataset.py", encoding="utf-8").read():
    raise RuntimeError("src/build_dataset.py on GitHub has no --relocate-from mode yet: push the latest commits")
run("src/build_dataset.py", "--relocate-from", BUILD, "--out", CFG)
assert os.path.isfile(f"{CFG}/data.yaml"), f"the relocation did not write {CFG}/data.yaml"
print(open(f"{CFG}/data.yaml").read())
print("Test-set yamls:", sorted(os.path.basename(p) for p in glob.glob(f"{CFG}/data_test_*.yaml")))


In [ ]:
# Resume: reuse the search log of a previous version of this notebook if one is attached as input.
# SEARCH_ARGS is used by the search cell below too, so the settings check here is exactly the one search.py makes.
import glob, json, shutil
sys.path.insert(0, "src")
from search import make_settings, parse_args
LOG, BEST = "results/gen2/search_pso.json", "results/gen2/best_pso.json"
SEARCH_ARGS = ["--method", "pso", "--data", f"{CFG}/data_proxy.yaml", "--imgsz", "640", "--log", LOG,
               "--best", BEST, "--project", "results/gen2/runs", "--device", "0"]
expected = make_settings(parse_args(SEARCH_ARGS))  # the settings search.py checks when it resumes
if os.path.exists(LOG):
    print(LOG, "already present: the search continues from it")
else:
    found = []
    for path in glob.glob(f"{INPUT}/**/{LOG}", recursive=True):
        try:
            previous = json.load(open(path))
        except (OSError, ValueError):
            print("Skipping unreadable", path)
            continue
        settings = previous.get("settings", {})
        diff = sorted(k for k in expected.keys() | settings.keys() if settings.get(k) != expected.get(k))
        if diff:  # e.g. a smoke run, another seed or imgsz: search.py would refuse it
            print(f"Skipping {path}: made with other settings ({', '.join(diff)})")
            continue
        found.append((len(previous.get("evaluations", [])), path))
    if found:
        n, path = max(found)  # if several versions are attached, the one that got furthest
        os.makedirs(os.path.dirname(LOG), exist_ok=True)
        shutil.copy(path, LOG)
        print(f"Resuming from {path} ({n} evaluations already done)")
    else:
        print("No usable previous log attached: starting a new search")


In [ ]:
# Stratified 40% subset of the train split (seed 42, stratified by source and dominant class)
# -> gen2_cfg/lists/train_proxy.txt and gen2_cfg/data_proxy.yaml
run("src/training.py", "--data", f"{CFG}/data.yaml", "--fraction", 0.4, "--split-csv", f"{CFG}/split.csv",
    "--out-yaml", f"{CFG}/data_proxy.yaml", "--out-list", f"{CFG}/lists/train_proxy.txt")
assert os.path.isfile(f"{CFG}/data_proxy.yaml"), "training.py did not write the proxy yaml"


In [ ]:
# One GPU (device 0), 12-epoch proxies at imgsz 640 on the 40% subset; the log is rewritten after every evaluation.
# After a 20-minute stall the watchdog restarts search.py with the same arguments: it replays the log and loses only
# the proxy training that was running.
run("src/search.py", *SEARCH_ARGS, watch=[LOG, "results/gen2/runs"])


In [ ]:
import json, os
LOG, BEST = "results/gen2/search_pso.json", "results/gen2/best_pso.json"
if not os.path.exists(LOG):
    print(f"{LOG} not found: the search wrote no evaluation (see the output of the search cell)")
else:
    log = json.load(open(LOG))
    s, evals = log["settings"], log["evaluations"]
    budget = s["n_particles"] * s["n_iters"]
    print(f"Evaluations done: {len(evals)} of {budget}" + ("" if log["complete"] else "  (not finished: rerun to continue)"))
    failed = sum(1 for e in evals if e.get("failed"))
    if failed:
        print(f"Failed trainings (recorded as mAP@50 0.0): {failed}")
    ok = [e for e in evals if not e.get("failed")]
    if ok:
        best = max(ok, key=lambda e: e["map50"])
        print(f"Best proxy val mAP@50: {best['map50']:.4f} at evaluation {best['index'] + 1} ({best['label']})")
        for name, value in best["params"].items():
            print(f"  {name:<13} {value:g}")
        hours = sum(e.get("seconds") or 0 for e in evals) / 3600
        print(f"Training time so far: {hours:.2f} h on {evals[-1].get('device')}")

    print("Best so far after each PSO round:")
    best_so_far = 0.0
    for r in range(1, s["n_iters"] + 1):
        scores = [e["map50"] for e in evals if e["round"] == r]
        if not scores:
            break
        best_so_far = max(best_so_far, *scores)
        note = "" if len(scores) == s["n_particles"] else f"  (round incomplete: {len(scores)} of {s['n_particles']})"
        print(f"  round {r} (evaluations up to {(r - 1) * s['n_particles'] + len(scores)}): {best_so_far:.4f}{note}")
    print(f"{BEST}:", "written" if os.path.exists(BEST) else "not written yet (only after the last evaluation)")


### Files to download / where they go locally

From the **Output** tab of the finished version:

| File in the Output tab | Where it goes locally |
|---|---|
| `SwarmSort/results/gen2/search_pso.json` | `results/gen2/search_pso.json` (every evaluation) |
| `SwarmSort/results/gen2/best_pso.json` | `results/gen2/best_pso.json` (the Arm C winner) |

The proxy run folders in `results/gen2/runs/` are not needed.

**Before running `g2_5_arm_c`:** commit `results/gen2/best_pso.json` (and the search log) and push, so the clone in
that notebook has it, **or** attach this notebook's output to `g2_5_arm_c` (its fallback cell copies the file from there).

### If the session died

**If a version stopped or froze, attach its output and run again: it continues.** Check the summary above: if
fewer than 32 evaluations are done, attach this version as input (Add Input -> Notebook
Output), run the notebook again with the same settings, and download the files from the version that finishes.